# Phase 2: exploring the images

**CRISP-DM stage:** Data understanding

**Runs on:** Kaggle, with the GPU and internet switched off, because no neural network runs and nothing is installed. The private dataset `revathyh/mvtec-ad2-sheet-metal-vial` is attached.

**Figures with dataset images:** files named `gallery_*` hold real dataset images, so they stay on my laptop and out of git. Each figure is saved and closed, not displayed, so no image ends up in the notebook's log.

## 1. Find the data

Same search as the Phase 1 notebook, see its part 1 for why we search and do not type a fixed path.

In [ ]:
from pathlib import Path

INPUT_FOLDER = Path("/kaggle/input")
PRODUCT_NAMES = ["sheet_metal", "vial"]

product_folders = {}

for train_folder in INPUT_FOLDER.rglob("train"):
    product_folder = train_folder.parent
    if product_folder.name in PRODUCT_NAMES:
        product_folders[product_folder.name] = product_folder

for product_name in PRODUCT_NAMES:
    if product_name not in product_folders:
        raise FileNotFoundError("Could not find the folder for " + product_name)
    print(product_name, "folder:", product_folders[product_name])

## 2. Helper functions

**Matplotlib** is Python's standard plotting library. A figure holds a grid of panels called axes, and each image goes in one panel.

- `pick_evenly` chooses files spread through a sorted folder. This shows more variety than the first few files, and it repeats exactly with no randomness.
- `draw_image` draws one greyscale image. `vmin=0` and `vmax=255` fix the brightness scale, so images can be compared. Matplotlib's default stretches each image's own range, which would make a dark image look normal.
- `contour` draws a line where the mask's values cross a level. 127.5 is halfway between 0 (no defect) and 255 (defect), so the line follows the defect's edge. An outline keeps the defect visible, where a filled overlay would hide it.

In [ ]:
import matplotlib.pyplot as plt
import numpy
from PIL import Image

# Sharp enough for the report, at a reasonable file size
FIGURE_DPI = 150


def pick_evenly(paths, how_many):
    """Return up to how_many paths, spread evenly through the list."""
    if len(paths) <= how_many:
        return paths
    picked = []
    step = len(paths) / how_many
    for position in range(how_many):
        picked.append(paths[int(position * step)])
    return picked


def read_image(path):
    """Read a PNG file into a table of numbers."""
    with Image.open(path) as image:
        return numpy.array(image)


def draw_image(axis, image_path, title, mask_path=None):
    """Draw one greyscale image in one panel, with its mask outlined if given."""
    axis.imshow(read_image(image_path), cmap="gray", vmin=0, vmax=255)
    if mask_path is not None:
        axis.contour(read_image(mask_path), levels=[127.5], colors="red", linewidths=1)
    axis.set_title(title, fontsize=10)
    axis.axis("off")


def mask_path_for(product_folder, image_path):
    """Return the mask file that belongs to a defective test image."""
    mask_name = image_path.stem + "_mask.png"
    return product_folder / "test_public" / "ground_truth" / "bad" / mask_name

## 3. Sample gallery

Three rows per product: good training images, good validation images, and defective test images with the mask outlined in red. There are 4 images per row, enough to see variety while staying legible. The defective row uses `regular` lighting, so it shows the defect and not a lighting change.

The figure size differs per product, because Sheet Metal is four times wider than tall and Vial is taller than wide.

**Limit:** the figure is shrunk to fit, so the tiniest defects may be hard to see, even with an outline.

In [ ]:
IMAGES_PER_ROW = 4
GALLERY_SIZES = {"sheet_metal": (24, 6), "vial": (16, 17)}

for product_name in PRODUCT_NAMES:
    product_folder = product_folders[product_name]

    train_paths = sorted((product_folder / "train" / "good").glob("*.png"))
    validation_paths = sorted((product_folder / "validation" / "good").glob("*.png"))
    defective_paths = sorted((product_folder / "test_public" / "bad").glob("*_regular.png"))

    rows = [
        ("train", pick_evenly(train_paths, IMAGES_PER_ROW)),
        ("validation", pick_evenly(validation_paths, IMAGES_PER_ROW)),
        ("defective test", pick_evenly(defective_paths, IMAGES_PER_ROW)),
    ]

    # squeeze=False keeps the panels in a table of rows and columns
    figure, axes = plt.subplots(3, IMAGES_PER_ROW, figsize=GALLERY_SIZES[product_name], squeeze=False)

    for row_number in range(3):
        row_name, paths = rows[row_number]
        for column_number in range(IMAGES_PER_ROW):
            axis = axes[row_number][column_number]
            # A row can have fewer images than panels on a small test sample
            if column_number >= len(paths):
                axis.axis("off")
                continue
            image_path = paths[column_number]
            mask_path = None
            if row_name == "defective test":
                mask_path = mask_path_for(product_folder, image_path)
            draw_image(axis, image_path, row_name + ": " + image_path.name, mask_path)

    figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0). Red outline: defect mask")
    file_name = "gallery_" + product_name + ".png"
    figure.savefig(file_name, dpi=FIGURE_DPI, bbox_inches="tight")
    # Closing without showing keeps dataset images out of the notebook's log
    plt.close(figure)
    print("Saved", file_name)

## 4. Scene check

File names suggest each number in `test_public` is one scene photographed under several lighting conditions. Here we look at defective scene `000` under every condition, with its masks outlined. This is one scene per product, so it can support the idea but cannot prove it for every scene.

In [ ]:
SCENE_NUMBER = "000"
SCENE_COLUMNS = {"sheet_metal": 2, "vial": 4}
SCENE_SIZES = {"sheet_metal": (24, 10), "vial": (16, 11)}

for product_name in PRODUCT_NAMES:
    product_folder = product_folders[product_name]
    scene_paths = sorted((product_folder / "test_public" / "bad").glob(SCENE_NUMBER + "_*.png"))

    columns = SCENE_COLUMNS[product_name]
    # Enough rows to hold every image, rounding up
    number_of_rows = (len(scene_paths) + columns - 1) // columns
    figure, axes = plt.subplots(number_of_rows, columns, figsize=SCENE_SIZES[product_name], squeeze=False)

    for row_number in range(number_of_rows):
        for column_number in range(columns):
            axis = axes[row_number][column_number]
            position = row_number * columns + column_number
            if position >= len(scene_paths):
                axis.axis("off")
                continue
            image_path = scene_paths[position]
            draw_image(axis, image_path, image_path.name, mask_path_for(product_folder, image_path))

    figure.suptitle("MVTec AD 2, " + product_name + ", scene " + SCENE_NUMBER + " (CC BY-NC-SA 4.0). Red outline: defect mask")
    file_name = "gallery_scene_check_" + product_name + ".png"
    figure.savefig(file_name, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(figure)
    print("Saved", file_name, "with", len(scene_paths), "images")

## 5. Public test set summary

This reads file names only, so no image is opened. It reports good and defective counts, scene numbers, lighting conditions, and whether the data labels defect types.

**Flag-everything baseline:** imagine a useless system that calls every image defective. It catches every defect but rejects every good part. With a mostly defective test set, it still scores well, so a method's F1 only means something if it beats this number. It uses only the counts, with no model.

In [ ]:
summary_lines = []

for product_name in PRODUCT_NAMES:
    test_folder = product_folders[product_name] / "test_public"
    good_paths = sorted((test_folder / "good").glob("*.png"))
    defective_paths = sorted((test_folder / "bad").glob("*.png"))

    good_count = len(good_paths)
    defective_count = len(defective_paths)
    total_count = good_count + defective_count

    summary_lines.append("")
    summary_lines.append("=== " + product_name)
    summary_lines.append("good images: " + str(good_count))
    summary_lines.append("defective images: " + str(defective_count))
    summary_lines.append("defective share: " + str(round(100 * defective_count / total_count, 1)) + " percent")

    # A file name such as 000_shift_1.png holds a scene number and a lighting condition
    good_scenes = []
    defective_scenes = []
    condition_counts = {}

    for path in good_paths:
        scene, condition = path.stem.split("_", 1)
        if scene not in good_scenes:
            good_scenes.append(scene)
        if condition not in condition_counts:
            condition_counts[condition] = [0, 0]
        condition_counts[condition][0] = condition_counts[condition][0] + 1

    for path in defective_paths:
        scene, condition = path.stem.split("_", 1)
        if scene not in defective_scenes:
            defective_scenes.append(scene)
        if condition not in condition_counts:
            condition_counts[condition] = [0, 0]
        condition_counts[condition][1] = condition_counts[condition][1] + 1

    summary_lines.append("good scene numbers: " + str(len(good_scenes)))
    summary_lines.append("defective scene numbers: " + str(len(defective_scenes)))

    summary_lines.append("lighting condition: good images, defective images")
    for condition in sorted(condition_counts):
        counts = condition_counts[condition]
        summary_lines.append("  " + condition + ": " + str(counts[0]) + ", " + str(counts[1]))

    # A defect type label would appear as a subfolder or in the file names
    subfolders = []
    for path in (test_folder / "bad").iterdir():
        if path.is_dir():
            subfolders.append(path.name)
    summary_lines.append("subfolders inside test_public/bad: " + str(len(subfolders)) + " " + str(subfolders))
    summary_lines.append("distinct name endings: " + ", ".join(sorted(condition_counts)))

    # Calling everything defective means no defect is missed, so recall is 1
    precision = defective_count / total_count
    recall = 1
    f1 = 2 * precision * recall / (precision + recall)
    accuracy = defective_count / total_count
    summary_lines.append("flag-everything baseline:")
    summary_lines.append("  accuracy: " + str(round(accuracy, 3)))
    summary_lines.append("  precision: " + str(round(precision, 3)))
    summary_lines.append("  recall: " + str(recall))
    summary_lines.append("  F1: " + str(round(f1, 3)))

# Numbers only, with no image content, so this file can be committed
with open("test_set_summary.txt", "w") as summary_file:
    for line in summary_lines:
        summary_file.write(line + "\n")

for line in summary_lines:
    print(line)

## 6. Split the public test set into half A and half B

**CRISP-DM stage:** Data preparation. This was planned for Phase 3. It moved here so that defect measurements can use half A only, and half B can never influence a choice.

Half A is for practice and for every choice that needs defect examples. Half B is the final exam, used only to report final results.

- **By scene number:** all lighting versions of one part go into the same half. Otherwise the exam would hold parts already seen.
- **Good and defective separately:** each half gets a fair share of both. Defect types are not labelled in the files, so they cannot be balanced.
- **Odd numbers:** an extra defective part goes to half B and an extra good part goes to half A, so the two halves come out as even as possible. For Vial this gives 70 photos in each half. The catch is that half B then has only 2 good Vial parts.
- **Seed 42:** a seed is a starting number that makes a shuffle repeat exactly. The value itself does not matter. The split is run once and never retried for a nicer result.

A library tool such as scikit-learn's splitters could do this too. A short shuffle is easier to explain line by line and does the same job here.

In [ ]:
import csv
import random

# Any fixed number repeats the same shuffle. It is never changed to get a nicer split
SPLIT_SEED = 42
random.seed(SPLIT_SEED)


def scene_of(file_name):
    """Return the scene number from a name such as 000_shift_1.png."""
    return file_name.split("_", 1)[0]


def split_scenes(scenes, label):
    """Shuffle scene numbers and share them out as evenly as possible."""
    shuffled = list(scenes)
    random.shuffle(shuffled)
    half_a_size = len(shuffled) // 2
    # An extra good part goes to half A and an extra defective part to half B,
    # because this makes the two halves as even in size as possible
    if label == "good":
        half_a_size = len(shuffled) - half_a_size
    return shuffled[:half_a_size], shuffled[half_a_size:]


def write_file_list(file_name, rows):
    """Save one half as a table of file names, with no image content."""
    with open(file_name, "w", newline="") as list_file:
        writer = csv.writer(list_file)
        writer.writerow(["product", "label", "scene", "file_name"])
        for row in rows:
            writer.writerow(row)


half_a_rows = []
half_b_rows = []

for product_name in PRODUCT_NAMES:
    test_folder = product_folders[product_name] / "test_public"
    for label in ["good", "bad"]:
        paths = sorted((test_folder / label).glob("*.png"))

        scenes = []
        for path in paths:
            scene = scene_of(path.name)
            if scene not in scenes:
                scenes.append(scene)

        half_a_scenes, half_b_scenes = split_scenes(scenes, label)

        half_a_images = 0
        half_b_images = 0
        for path in paths:
            scene = scene_of(path.name)
            row = [product_name, label, scene, path.name]
            if scene in half_a_scenes:
                half_a_rows.append(row)
                half_a_images = half_a_images + 1
            else:
                half_b_rows.append(row)
                half_b_images = half_b_images + 1

        print(product_name, label, "half A:", len(half_a_scenes), "parts,", half_a_images, "images",
              "| half B:", len(half_b_scenes), "parts,", half_b_images, "images")

# A safety check: the same part must never appear in both halves
half_b_parts = []
for row in half_b_rows:
    half_b_parts.append((row[0], row[1], row[2]))
for row in half_a_rows:
    if (row[0], row[1], row[2]) in half_b_parts:
        raise ValueError("A part is in both halves: " + str(row))

write_file_list("half_a_files.csv", half_a_rows)
write_file_list("half_b_files.csv", half_b_rows)
print("Saved half_a_files.csv and half_b_files.csv")

## 7. Measure every defect in half A

**CRISP-DM stage:** Data understanding. Only half A is read here, so half B stays sealed.

A mask is a black-and-white copy of a photo where white marks the defect. One mask can show several separate defects, so we find each separate white shape.

- `ndimage.label` gives every separate white shape its own number.
- `structure` says which pixels count as touching. The 3 by 3 block of ones includes corners. The tool's default counts only side-by-side touching, which would break a diagonal scratch into many tiny defects. anomalib's code for its AU-PRO score uses OpenCV's `connectedComponents`, whose default also counts corners, so this matches.
- `ndimage.find_objects` gives the smallest box around each shape, which gives its width and height.

The small test first uses a made-up mask with a known answer, so a mistake in the counting shows up straight away.

**Repeated photos:** one defect appears once per lighting condition, so the number of defects is larger than the number of physical parts. Every photo is kept because some masks differ between lighting conditions.

**Limit:** width and height come from a box drawn around the defect. A thin diagonal scratch has a large box, so it can look wider than it really is.

In [ ]:
from scipy import ndimage

# A 3 by 3 block of ones means pixels touching at corners count as one defect
TOUCHING_RULE = numpy.ones((3, 3))


def find_defects(mask):
    """Return the area, width and height of each separate white shape in a mask."""
    numbered_mask, number_of_defects = ndimage.label(mask > 0, structure=TOUCHING_RULE)
    boxes = ndimage.find_objects(numbered_mask)

    defects = []
    for position in range(number_of_defects):
        box = boxes[position]
        row_range = box[0]
        column_range = box[1]
        # Shapes are numbered from 1. A box may also hold part of a neighbouring shape
        area = numpy.sum(numbered_mask[box] == position + 1)
        defects.append({
            "area": int(area),
            "width": column_range.stop - column_range.start,
            "height": row_range.stop - row_range.start,
        })
    return defects


# Made-up mask with a known answer: a 2 by 2 square, a 3-pixel diagonal and a single pixel
test_mask = numpy.zeros((6, 6))
test_mask[0:2, 0:2] = 255
test_mask[3, 3] = 255
test_mask[4, 4] = 255
test_mask[5, 5] = 255
test_mask[0, 5] = 255

test_areas = []
for defect in find_defects(test_mask):
    test_areas.append(defect["area"])
if sorted(test_areas) != [1, 3, 4]:
    raise ValueError("Defect counting is wrong: " + str(test_areas))
print("Counting test passed, areas found:", sorted(test_areas))

defect_rows = []
image_sizes = {}

for row in half_a_rows:
    product_name, label, scene, file_name = row
    # Good images have no masks, so only defective images are measured
    if label != "bad":
        continue

    image_path = product_folders[product_name] / "test_public" / "bad" / file_name
    mask = read_image(mask_path_for(product_folders[product_name], image_path))
    height, width = mask.shape
    image_sizes[product_name] = (width, height)
    image_area = width * height

    lighting = Path(file_name).stem.split("_", 1)[1]
    for defect in find_defects(mask):
        defect_rows.append({
            "product": product_name,
            "file_name": file_name,
            "scene": scene,
            "lighting": lighting,
            "area_pixels": defect["area"],
            "area_percent": 100 * defect["area"] / image_area,
            "width": defect["width"],
            "height": defect["height"],
        })

with open("defect_sizes.csv", "w", newline="") as sizes_file:
    writer = csv.DictWriter(sizes_file, fieldnames=list(defect_rows[0].keys()))
    writer.writeheader()
    for defect_row in defect_rows:
        writer.writerow(defect_row)

print("Saved defect_sizes.csv with", len(defect_rows), "defects")

## 8. How small do defects become when an image is shrunk?

This is arithmetic on the measured sizes. No image is actually shrunk, and no image size is chosen here.

If an image becomes 16 times narrower and 16 times shorter, a defect's area becomes 16 × 16 = 256 times smaller.

- **256 and 512 wide** are the two sizes in the question.
- **Keep shape:** width and height shrink by the same amount, so the part is not distorted.
- **Squash to square:** the image becomes 256 by 256 or 512 by 512. Width and height shrink by different amounts. This is included because the shape choice is not made until Phase 3.

A defect below 1 pixel of area, or narrower than 1 pixel in either direction, may disappear when shrunk.

In [ ]:
TARGET_SIZES = [256, 512]

size_lines = []

for product_name in PRODUCT_NAMES:
    areas = []
    percents = []
    parts = []
    images = []
    product_rows = []
    for defect_row in defect_rows:
        if defect_row["product"] == product_name:
            product_rows.append(defect_row)
            areas.append(defect_row["area_pixels"])
            percents.append(defect_row["area_percent"])
            if defect_row["scene"] not in parts:
                parts.append(defect_row["scene"])
            if defect_row["file_name"] not in images:
                images.append(defect_row["file_name"])

    width, height = image_sizes[product_name]

    size_lines.append("")
    size_lines.append("=== " + product_name + ", half A only")
    size_lines.append("original image: " + str(width) + " x " + str(height))
    size_lines.append("defective parts: " + str(len(parts)))
    size_lines.append("defective images: " + str(len(images)))
    size_lines.append("separate defects found: " + str(len(areas)))
    size_lines.append("area in pixels, smallest: " + str(min(areas)))
    size_lines.append("area in pixels, middle (median): " + str(numpy.median(areas)))
    size_lines.append("area in pixels, largest: " + str(max(areas)))
    size_lines.append("area in percent of image, smallest: " + str(round(min(percents), 5)))
    size_lines.append("area in percent of image, middle (median): " + str(round(float(numpy.median(percents)), 5)))
    size_lines.append("area in percent of image, largest: " + str(round(max(percents), 5)))

    for target in TARGET_SIZES:
        # The two ways of shrinking: same scale both ways, or squashed into a square
        shrink_options = [
            ("keep shape, " + str(target) + " wide", target / width, target / width),
            ("squash to " + str(target) + " x " + str(target), target / width, target / height),
        ]
        for option_name, width_scale, height_scale in shrink_options:
            below_one_area = 0
            below_one_side = 0
            for defect_row in product_rows:
                new_area = defect_row["area_pixels"] * width_scale * height_scale
                new_width = defect_row["width"] * width_scale
                new_height = defect_row["height"] * height_scale
                if new_area < 1:
                    below_one_area = below_one_area + 1
                if new_width < 1 or new_height < 1:
                    below_one_side = below_one_side + 1

            size_lines.append(option_name + ":")
            size_lines.append("  one shrunk pixel covers " + str(round(1 / (width_scale * height_scale), 1)) + " original pixels")
            size_lines.append("  smallest defect becomes " + str(round(min(areas) * width_scale * height_scale, 3)) + " pixels")
            size_lines.append("  middle defect becomes " + str(round(float(numpy.median(areas)) * width_scale * height_scale, 3)) + " pixels")
            size_lines.append("  defects below 1 pixel of area: " + str(below_one_area) + " of " + str(len(areas)))
            size_lines.append("  defects narrower than 1 pixel in width or height: " + str(below_one_side) + " of " + str(len(areas)))

# Numbers only, with no image content, so this file can be committed
with open("defect_size_summary.txt", "w") as summary_file:
    for line in size_lines:
        summary_file.write(line + "\n")

for line in size_lines:
    print(line)

## 9. Defect-size histogram

A histogram is a bar chart of how many defects fall into each size range.

- **Stretched scale:** each step along the bottom is 10 times bigger (1, 10, 100, 1,000...). Defect sizes vary hugely, and an ordinary scale would squash all the tiny ones into one bar.
- **Four bars per step of 10:** enough to show the shape without single defects looking like a pattern.
- **Same scale in both panels,** so the two products can be compared.
- **Dashed lines** mark how many original pixels fit into one shrunk pixel at 256 and 512 wide, keeping the shape. A defect left of a line would be smaller than one pixel at that size.

The figure holds numbers only, with no dataset images, so it can be committed.

In [ ]:
from matplotlib.ticker import MaxNLocator

# From 1 pixel up to 10 million, which is larger than either whole image
SIZE_BINS = numpy.logspace(0, 7, 29)

BAR_COLOUR = "#2a78d6"
LINE_COLOUR = "#3a3a38"
# Different heights stop the two line labels from overlapping
LABEL_HEIGHTS = {256: 0.97, 512: 0.87}

figure, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

for position in range(len(PRODUCT_NAMES)):
    product_name = PRODUCT_NAMES[position]
    axis = axes[position]

    areas = []
    parts = []
    for defect_row in defect_rows:
        if defect_row["product"] == product_name:
            areas.append(defect_row["area_pixels"])
            if defect_row["scene"] not in parts:
                parts.append(defect_row["scene"])

    axis.hist(areas, bins=SIZE_BINS, color=BAR_COLOUR, edgecolor="white", linewidth=1.5)
    axis.set_xscale("log")
    axis.set_ylabel("Number of defects")
    axis.set_title(product_name + ": " + str(len(areas)) + " defects from " + str(len(parts)) + " parts, half A only", loc="left")
    # You cannot have part of a defect, so the side axis uses whole numbers
    axis.yaxis.set_major_locator(MaxNLocator(integer=True))
    axis.grid(axis="y", color="#e4e4e0", linewidth=0.8)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

    width, height = image_sizes[product_name]
    for target in TARGET_SIZES:
        # How many original pixels fit into one shrunk pixel, keeping the shape
        one_pixel_area = (width / target) * (width / target)
        axis.axvline(one_pixel_area, color=LINE_COLOUR, linestyle="--", linewidth=1)
        # Places the label at the top of the panel, whatever the bar heights are
        axis.text(one_pixel_area, LABEL_HEIGHTS[target], " 1 pixel at " + str(target) + " wide",
                  transform=axis.get_xaxis_transform(), fontsize=9, color=LINE_COLOUR, va="top")

axes[1].set_xlabel("Defect area in original pixels (each step is 10 times bigger)")
figure.tight_layout()
figure.savefig("defect_size_histogram.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(figure)
print("Saved defect_size_histogram.png")

## 10. Are defects near the borders, and what would cropping lose?

**CRISP-DM stage:** Data understanding. Only half A is read here, so half B stays sealed. No crop is chosen here, and no image is changed.

Cropping means cutting off the edges of a photo. A defect that is cut off can never be found, so we check what each crop would remove.

**Near the border:** there is no official meaning of "near", so 5 and 10 percent of the photo's width or height are working choices. Showing two reveals whether the answer depends on the choice.

**Crops tested:**
- **Square from the centre:** keeps a square as wide as the shorter side, because this is a common way to make a photo square.
- **Central 90 percent:** trims 5 percent from each side, as a gentle crop for comparison.

Same defect counting as part 7. The small test uses a made-up mask with a known answer.

**Limit:** the distance is measured from the photo's edge, and the part may not fill the photo. A defect can be near the part's edge but far from the photo's edge.

In [ ]:
NEAR_SHARES = [0.05, 0.10]
CROP_NAMES = ["square from the centre", "central 90 percent"]
# 5 percent from each side leaves the central 90 percent
GENTLE_TRIM = 0.05


def crop_edges(width, height, crop_name):
    """Return the left, right, top and bottom edges of the area a crop keeps."""
    if crop_name == "square from the centre":
        side = min(width, height)
        left = (width - side) // 2
        top = (height - side) // 2
        return left, left + side, top, top + side
    left = round(width * GENTLE_TRIM)
    top = round(height * GENTLE_TRIM)
    return left, width - left, top, height - top


def describe_locations(mask):
    """Return where each separate defect in a mask sits, and what each crop keeps."""
    height, width = mask.shape
    numbered_mask, number_of_defects = ndimage.label(mask > 0, structure=TOUCHING_RULE)
    boxes = ndimage.find_objects(numbered_mask)

    defects = []
    for position in range(number_of_defects):
        defect_number = position + 1
        row_range = boxes[position][0]
        column_range = boxes[position][1]

        from_left = column_range.start
        from_right = width - column_range.stop
        from_top = row_range.start
        from_bottom = height - row_range.stop

        near = {}
        for share in NEAR_SHARES:
            # Width and height use their own share, because the photos are not square
            close_sideways = min(from_left, from_right) < share * width
            close_up_or_down = min(from_top, from_bottom) < share * height
            near[share] = close_sideways or close_up_or_down

        area = numpy.sum(numbered_mask[boxes[position]] == defect_number)
        kept = {}
        for crop_name in CROP_NAMES:
            left, right, top, bottom = crop_edges(width, height, crop_name)
            kept[crop_name] = numpy.sum(numbered_mask[top:bottom, left:right] == defect_number)

        defects.append({
            "touches": min(from_left, from_right, from_top, from_bottom) == 0,
            "near": near,
            "area": int(area),
            "kept": kept,
        })
    return defects


# Made-up mask, 40 wide and 20 tall. The square crop keeps columns 10 to 29
test_mask = numpy.zeros((20, 40))
test_mask[8:10, 0:2] = 255     # touches the left edge, and the square crop loses all of it
test_mask[9:11, 19:21] = 255   # in the centre, and fully kept
test_mask[5:7, 9:11] = 255     # the crop's edge cuts this one in half

test_defects = describe_locations(test_mask)
test_touching = 0
test_lost = 0
test_cut = 0
test_kept_pixels = 0
for defect in test_defects:
    kept_pixels = defect["kept"]["square from the centre"]
    test_kept_pixels = test_kept_pixels + kept_pixels
    if defect["touches"]:
        test_touching = test_touching + 1
    if kept_pixels == 0:
        test_lost = test_lost + 1
    if 0 < kept_pixels < defect["area"]:
        test_cut = test_cut + 1
if [len(test_defects), test_touching, test_lost, test_cut, test_kept_pixels] != [3, 1, 1, 1, 6]:
    raise ValueError("Border or crop counting is wrong")
print("Border and crop test passed")

location_lines = []
location_maps = {}

for product_name in PRODUCT_NAMES:
    image_count = 0
    defect_count = 0
    touching = 0
    near_counts = {}
    for share in NEAR_SHARES:
        near_counts[share] = 0
    near_parts = []
    touching_parts = []
    all_parts = []
    total_pixels = 0
    crop_results = {}
    for crop_name in CROP_NAMES:
        crop_results[crop_name] = {"lost": 0, "cut": 0, "kept_pixels": 0}
    location_map = None

    for row in half_a_rows:
        row_product, label, scene, file_name = row
        # Good images have no masks, and half B is never read here
        if row_product != product_name or label != "bad":
            continue

        image_path = product_folders[product_name] / "test_public" / "bad" / file_name
        mask = read_image(mask_path_for(product_folders[product_name], image_path))

        # Adds this mask to the stack: each spot counts the photos with a defect there
        if location_map is None:
            location_map = numpy.zeros(mask.shape)
        location_map = location_map + (mask > 0)
        image_count = image_count + 1
        if scene not in all_parts:
            all_parts.append(scene)

        for defect in describe_locations(mask):
            defect_count = defect_count + 1
            total_pixels = total_pixels + defect["area"]
            if defect["touches"]:
                touching = touching + 1
                if scene not in touching_parts:
                    touching_parts.append(scene)
            for share in NEAR_SHARES:
                if defect["near"][share]:
                    near_counts[share] = near_counts[share] + 1
            if defect["near"][0.10] and scene not in near_parts:
                near_parts.append(scene)
            for crop_name in CROP_NAMES:
                kept_pixels = defect["kept"][crop_name]
                crop_results[crop_name]["kept_pixels"] = crop_results[crop_name]["kept_pixels"] + kept_pixels
                if kept_pixels == 0:
                    crop_results[crop_name]["lost"] = crop_results[crop_name]["lost"] + 1
                elif kept_pixels < defect["area"]:
                    crop_results[crop_name]["cut"] = crop_results[crop_name]["cut"] + 1

    # Dividing by the photo count gives the share of photos with a defect at each spot
    location_maps[product_name] = location_map / image_count

    location_lines.append("")
    location_lines.append("=== " + product_name + ", half A only")
    location_lines.append("defective parts: " + str(len(all_parts)))
    location_lines.append("defective images: " + str(image_count))
    location_lines.append("separate defects: " + str(defect_count))
    location_lines.append("highest share of photos with a defect at one spot: " + str(round(float(location_maps[product_name].max()), 3)))
    location_lines.append("defects touching a photo edge: " + str(touching) + " of " + str(defect_count)
                          + ", from " + str(len(touching_parts)) + " of " + str(len(all_parts)) + " parts")
    for share in NEAR_SHARES:
        location_lines.append("defects within " + str(round(share * 100)) + " percent of a photo edge: "
                              + str(near_counts[share]) + " of " + str(defect_count))
    location_lines.append("parts with a defect within 10 percent of a photo edge: " + str(len(near_parts)) + " of " + str(len(all_parts)))

    height, width = location_map.shape
    for crop_name in CROP_NAMES:
        left, right, top, bottom = crop_edges(width, height, crop_name)
        results = crop_results[crop_name]
        lost_share = 100 * (total_pixels - results["kept_pixels"]) / total_pixels
        location_lines.append(crop_name + ", keeping " + str(right - left) + " x " + str(bottom - top) + ":")
        location_lines.append("  defects fully lost: " + str(results["lost"]) + " of " + str(defect_count))
        location_lines.append("  defects partly cut: " + str(results["cut"]) + " of " + str(defect_count))
        location_lines.append("  defect pixels lost: " + str(round(float(lost_share), 1)) + " percent")

# Numbers only, with no image content, so this file can be committed
with open("defect_location_summary.txt", "w") as summary_file:
    for line in location_lines:
        summary_file.write(line + "\n")

for line in location_lines:
    print(line)

## 11. Defect location maps

Imagine drawing each defect on a clear sheet and stacking the sheets. Spots where many defects overlap look dark.

- **Each spot** shows the share of half A's defective photos with a defect there. One colour from light to dark is used, because the only thing shown is how much.
- **The dashed box** shows what a square crop from the centre would keep.
- `imshow` shrinks the map to fit the figure using matplotlib's default smoothing, so the tiniest defects may be too faint to see. The summary numbers above still count them.

With only 7 parts per product, a map could look almost like the masks themselves, so these files are named `gallery_*` and stay out of git.

In [ ]:
from matplotlib.patches import Rectangle

MAP_SIZES = {"sheet_metal": (16, 5.5), "vial": (9, 10)}

for product_name in PRODUCT_NAMES:
    location_map = location_maps[product_name]
    height, width = location_map.shape

    figure, axis = plt.subplots(figsize=MAP_SIZES[product_name])
    # vmin=0 keeps "no defect ever" white, not just the lowest value found
    picture = axis.imshow(location_map, cmap="Blues", vmin=0)
    figure.colorbar(picture, ax=axis, label="Share of defective photos with a defect here", shrink=0.8)

    left, right, top, bottom = crop_edges(width, height, "square from the centre")
    crop_box = Rectangle((left, top), right - left, bottom - top, fill=False,
                         edgecolor="#3a3a38", linestyle="--", linewidth=1.2)
    axis.add_patch(crop_box)

    axis.set_title("MVTec AD 2 (CC BY-NC-SA 4.0), " + product_name
                   + ": where defects appear, half A only. Dashed box: square crop from the centre", fontsize=10)
    axis.set_xlabel("Pixels across the photo")
    axis.set_ylabel("Pixels down the photo")

    file_name = "gallery_defect_location_" + product_name + ".png"
    figure.savefig(file_name, dpi=FIGURE_DPI, bbox_inches="tight")
    # Closing without showing keeps the map out of the notebook's log
    plt.close(figure)
    print("Saved", file_name)

## 12. How much do good parts vary?

**CRISP-DM stage:** Data understanding. Only good training photos are read here, so half B is not involved.

Imagine laying all the good photos on top of each other like see-through sheets.

- **Average image:** what the stack looks like. Parts in the same spot look sharp, and parts that move look blurry.
- **Variation:** how far each pixel typically is from its own average, called the standard deviation. It is measured in brightness units.

Photos are read one at a time, and running totals are kept, because this is enough to work out the average and variation. Loading every photo at once would give the same answer but use far more memory.

The small test uses two tiny made-up pictures with a known answer.

In [ ]:
def average_and_variation(total, total_of_squares, photo_count):
    """Work out each pixel's average and standard deviation from running totals."""
    average = total / photo_count
    squared_spread = total_of_squares / photo_count - average * average
    # Rounding can leave a tiny negative number, which a square root cannot use
    squared_spread = numpy.maximum(squared_spread, 0)
    return average, numpy.sqrt(squared_spread)


# Two made-up pictures. Known answer: average [5, 10] and variation [5, 0]
test_pictures = [numpy.array([[0.0, 10.0]]), numpy.array([[10.0, 10.0]])]
test_total = numpy.zeros((1, 2))
test_squares = numpy.zeros((1, 2))
for picture in test_pictures:
    test_total = test_total + picture
    test_squares = test_squares + picture * picture
test_average, test_variation = average_and_variation(test_total, test_squares, 2)
if test_average.tolist() != [[5.0, 10.0]] or test_variation.tolist() != [[5.0, 0.0]]:
    raise ValueError("Average or variation is wrong")
print("Average and variation test passed")

# Every 4th pixel is kept for the movement check in the next part
POSITION_STEP = 4

average_images = {}
variation_images = {}
small_photos = {}
photo_names = {}

for product_name in PRODUCT_NAMES:
    train_paths = sorted((product_folders[product_name] / "train" / "good").glob("*.png"))
    total = None
    total_of_squares = None
    small_photos[product_name] = []
    photo_names[product_name] = []

    for path in train_paths:
        # float64 keeps the running totals exact enough, and cannot overflow here
        photo = read_image(path).astype(numpy.float64)
        if total is None:
            total = numpy.zeros(photo.shape)
            total_of_squares = numpy.zeros(photo.shape)
        total = total + photo
        total_of_squares = total_of_squares + photo * photo
        small_photos[product_name].append(photo[::POSITION_STEP, ::POSITION_STEP])
        photo_names[product_name].append(path.name)

    average, variation = average_and_variation(total, total_of_squares, len(train_paths))
    average_images[product_name] = average
    variation_images[product_name] = variation
    print(product_name, "photos used:", len(train_paths))

## 13. Are the parts always in the same position?

A map alone cannot tell a moving part from a changing surface, so this measures movement in pixels.

`phase_cross_correlation` from scikit-image slides each photo over a reference and finds the shift where they match best.

- `reference_image` is the average of all the other photos, so a photo can never partly match itself.
- `moving_image` is the photo being checked.
- `normalization=None` means plain matching, which compares the overall picture. The tool's default, `"phase"`, reported no movement for a sample vial that had clearly moved, because it pays close attention to fine patterns.
- We rely on its default `upsample_factor=1`, which finds whole steps only.
- **Every 4th pixel only:** this is about 16 times faster. One step equals 4 original pixels, which is accurate enough to see whether parts move.

Finding the part's outline with a brightness cut-off was not used, because backgrounds and lighting can vary. Looking by eye was not enough either, because it gives no numbers.

**10 and 50 pixels** are working choices that make the answer easy to read.

**Limits:** this only measures sliding, across the whole photo. It does not measure turning, such as a tilted lid, or one piece moving separately, such as a label. The background does not move, so it can pull answers towards 0. The results may understate movement.

Two small tests have known answers. One moves a real photo by a known amount. The other uses Vial photo `003`, which measuring its edges showed sits about 30 pixels left of the other sample vials, so it must not come back as no movement.

In [ ]:
from skimage.registration import phase_cross_correlation

MOVEMENT_LIMITS = [10, 50]


def find_shift(reference, photo):
    """Return how many original pixels a photo must move down and across to match."""
    # The tool also returns two other values, which are not needed here
    # normalization=None is plain matching. The default setting missed a real movement
    shift, error, phase_difference = phase_cross_correlation(reference, photo, normalization=None)
    return int(shift[0]) * POSITION_STEP, int(shift[1]) * POSITION_STEP


# Known answer: rolling a photo down 5 steps and left 8 steps needs a shift of -5 and +8
test_photo = small_photos[PRODUCT_NAMES[0]][0]
moved_photo = numpy.roll(test_photo, (5, -8), axis=(0, 1))
test_down, test_across = find_shift(test_photo, moved_photo)
if test_down != -5 * POSITION_STEP or test_across != 8 * POSITION_STEP:
    raise ValueError("Shift check is wrong: " + str(test_down) + ", " + str(test_across))
print("Shift test passed:", test_down, test_across)


def average_of_others(photos, total, position):
    """Return the average of every photo except one, so it cannot match itself."""
    return (total - photos[position]) / (len(photos) - 1)


# Known real case: this vial sits to the left of the others, so it needs to move right
vial_photos = small_photos["vial"]
vial_total = numpy.zeros(vial_photos[0].shape)
for photo in vial_photos:
    vial_total = vial_total + photo
vial_position = photo_names["vial"].index("003_regular.png")
vial_down, vial_across = find_shift(average_of_others(vial_photos, vial_total, vial_position), vial_photos[vial_position])
# 20 is safely below the roughly 30 pixels measured from the vial's edges
if vial_across < 20:
    raise ValueError("Movement check missed the known moved vial: " + str(vial_across))
print("Known moved vial found, sideways shift:", vial_across)

position_rows = []
variation_lines = []

for product_name in PRODUCT_NAMES:
    product_photos = small_photos[product_name]
    product_total = numpy.zeros(product_photos[0].shape)
    for photo in product_photos:
        product_total = product_total + photo
    down_sizes = []
    across_sizes = []
    largest_sizes = []

    for position in range(len(product_photos)):
        reference = average_of_others(product_photos, product_total, position)
        down, across = find_shift(reference, product_photos[position])
        position_rows.append([product_name, photo_names[product_name][position], down, across])
        down_sizes.append(abs(down))
        across_sizes.append(abs(across))
        largest_sizes.append(max(abs(down), abs(across)))

    variation = variation_images[product_name]
    variation_lines.append("")
    variation_lines.append("=== " + product_name + ", good training photos only")
    variation_lines.append("photos used: " + str(len(largest_sizes)))
    variation_lines.append("variation in brightness units (0 to 255 scale):")
    variation_lines.append("  typical pixel (median): " + str(round(float(numpy.median(variation)), 1)))
    variation_lines.append("  average pixel: " + str(round(float(numpy.mean(variation)), 1)))
    variation_lines.append("  most varying 1 percent start at: " + str(round(float(numpy.percentile(variation, 99)), 1)))
    variation_lines.append("  highest pixel: " + str(round(float(numpy.max(variation)), 1)))
    variation_lines.append("movement needed to match the other photos, in original pixels:")
    variation_lines.append("  up or down, median: " + str(numpy.median(down_sizes)) + ", largest: " + str(max(down_sizes)))
    variation_lines.append("  sideways, median: " + str(numpy.median(across_sizes)) + ", largest: " + str(max(across_sizes)))
    for limit in MOVEMENT_LIMITS:
        moved_count = 0
        for size in largest_sizes:
            if size > limit:
                moved_count = moved_count + 1
        variation_lines.append("  photos moved more than " + str(limit) + " pixels: " + str(moved_count) + " of " + str(len(largest_sizes)))

with open("good_part_positions.csv", "w", newline="") as positions_file:
    writer = csv.writer(positions_file)
    writer.writerow(["product", "file_name", "shift_down_pixels", "shift_across_pixels"])
    for row in position_rows:
        writer.writerow(row)

# Numbers only, with no image content, so these files can be committed
with open("good_part_variation_summary.txt", "w") as summary_file:
    for line in variation_lines:
        summary_file.write(line + "\n")

for line in variation_lines:
    print(line)

## 14. Average image and variation map

- **Average image, left:** uses the same 0 to 255 brightness scale as earlier figures.
- **Variation map, right:** bright means this spot changes a lot between photos. The scale runs from 0 to 127.5, which is the largest standard deviation possible on brightness from 0 to 255. Both products use this same scale, so they can be compared.

An average image still looks like the dataset's parts, so these files are named `gallery_*` and stay out of git.

In [ ]:
# The largest possible standard deviation: half the photos black, half white
LARGEST_VARIATION = 127.5

VARIATION_LAYOUT = {"sheet_metal": (2, 1, (16, 9)), "vial": (1, 2, (13, 9))}

for product_name in PRODUCT_NAMES:
    rows, columns, figure_size = VARIATION_LAYOUT[product_name]
    figure, axes = plt.subplots(rows, columns, figsize=figure_size)

    axes[0].imshow(average_images[product_name], cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Average of good training photos", fontsize=11)
    axes[0].axis("off")

    picture = axes[1].imshow(variation_images[product_name], cmap="gray", vmin=0, vmax=LARGEST_VARIATION)
    axes[1].set_title("How much each spot varies (bright means more)", fontsize=11)
    axes[1].axis("off")
    figure.colorbar(picture, ax=axes[1], label="Standard deviation of brightness", shrink=0.8)

    figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0)")
    file_name = "gallery_good_variation_" + product_name + ".png"
    figure.savefig(file_name, dpi=FIGURE_DPI, bbox_inches="tight")
    # Closing without showing keeps dataset-like images out of the notebook's log
    plt.close(figure)
    print("Saved", file_name)